# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dv-06/flyrank-ml_1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a Random Forest model because it can learn patterns from multiple features without relying on a single rule. It is suitable for this project because webpage performance depends on several factors, such as impressions, CTR, average position, and content age.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)
print("Number of Clients:", df["client_id"].nunique())
print("Declining Pages:",
      (df["trend_direction"].str.lower() == "down").sum())
print("Declining Rate:",
      round((df["trend_direction"].str.lower() == "down").mean(), 3))

Dataset Shape: (30000, 44)
Number of Clients: 32
Declining Pages: 16262
Declining Rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a client-based split so that pages from the same client do not appear in both the training and testing data. This gives a more realistic evaluation because the model is tested on unseen client data instead of data it has already seen.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import train_test_split

# Create target column
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Features
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Feature matrix and target
X = df[features].fillna(0)
y = df["is_declining"]

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features used:", len(features))
print(features)

Training rows: 24000
Testing rows: 6000
Features used: 15
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will compare the Random Forest model with my baseline rule using the same dataset and evaluation metric. This helps me understand whether the machine learning model provides better recommendations than the simple rule-based approach.

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC AUC": roc_auc_score(y_test, probabilities)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="ROC AUC",
    ascending=False
)

results_df.round(3)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy,Precision,Recall,F1,ROC AUC
2,Random Forest,0.696,0.704,0.759,0.731,0.762
1,Decision Tree,0.626,0.657,0.647,0.652,0.624
0,Logistic Regression,0.613,0.613,0.776,0.685,0.615


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Some pages may have similar feature values but different outcomes, which can lead to incorrect predictions. The model should be used to support editorial decisions and not replace human review.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Best model based on ROC AUC
best_model_name = results_df.iloc[0]["Model"]
print("Best Model:", best_model_name)

# Random Forest feature importance
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 10 Important Features")
print(importance.head(10))

# Error Analysis
predictions = rf.predict(X_test)

errors = X_test.copy()
errors["Actual"] = y_test.values
errors["Predicted"] = predictions

wrong = errors[errors["Actual"] != errors["Predicted"]]

print("\nNumber of Incorrect Predictions:", len(wrong))
print("\nSample Incorrect Predictions")
wrong.head(10)

Best Model: Random Forest

Top 10 Important Features
             Feature  Importance
5    impressions_90d    0.157065
11      avg_position    0.148735
8   content_age_days    0.102826
4         char_count    0.082508
3         word_count    0.082453
7       sessions_90d    0.073967
10               ctr    0.060913
13       scroll_rate    0.060482
6         clicks_90d    0.047842
0      search_volume    0.042219

Number of Incorrect Predictions: 1821

Sample Incorrect Predictions


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,Actual,Predicted
2110,0.0,0.00,0.00,6466.0,42912.0,511,2,41,223,104,0.39,15.0,0.00,4.55,0.00,0,1
27201,20.0,0.13,0.00,4162.0,24550.0,66,0,2,310,104,0.00,45.5,0.00,50.00,0.00,1,0
25967,0.0,0.00,0.00,0.0,0.0,373,0,1,502,20,0.00,60.4,0.00,0.00,100.00,1,0
26973,10.0,0.00,0.00,2602.0,18454.0,595,1,2,112,20,0.17,5.2,0.00,0.00,0.00,0,1
12628,0.0,0.00,0.00,4583.0,29476.0,40521,46,410,211,104,0.11,32.7,1.46,9.85,0.98,1,0
28981,20.0,0.42,2.54,2443.0,14702.0,16902,71,84,310,26,0.42,10.7,3.57,5.71,0.00,0,1
13530,390.0,0.47,0.38,0.0,0.0,158,0,1,460,22,0.00,23.4,0.00,0.00,0.00,1,0
17195,0.0,0.00,0.00,5572.0,37798.0,3131,1,35,139,104,0.03,21.2,0.00,0.00,0.00,0,1
19583,10.0,0.23,0.04,2689.0,17896.0,1103,1,4,175,20,0.09,9.6,0.00,20.00,0.00,0,1
17941,10.0,0.39,0.00,2363.0,16377.0,510,3,7,112,20,0.59,12.0,0.00,0.00,0.00,0,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.